In [1]:
# Lets start by establishing project paths and basic enviroment to ensure all our work is safe,


from pathlib import Path
import pandas as pd
import requests
from bs4 import BeautifulSoup

# Project root
PROJECT_ROOT = Path.cwd().parent

# Data directories
RAW_DIR = PROJECT_ROOT / "data" / "raw"
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"

# Create directories if they don't already exist
RAW_DIR.mkdir(parents=True, exist_ok=True)
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

print("Project root:", PROJECT_ROOT)
print("Raw data directory:", RAW_DIR)
print("Processed data directory:", PROCESSED_DIR)

Project root: C:\Users\Student\Downloads\Big Data\SPU-TEAM-DIRISA
Raw data directory: C:\Users\Student\Downloads\Big Data\SPU-TEAM-DIRISA\data\raw
Processed data directory: C:\Users\Student\Downloads\Big Data\SPU-TEAM-DIRISA\data\processed


In [2]:
# Okay, let's start the extraction from Data Source 1:
# IEC Voter Registration Statistics.
#
# We use collecting till the current year because voter registration is a continuous
# activity rather than something that only happens during elections.
# We start from 2011 because this gives us a relevant historical period
# for analysing changes in voter registration over time.

IEC_VOTER_REGISTRATION_URL = (
    "https://www.elections.org.za/pw/StatsData/Voter-Registration-Statistics"
)

# Project scope
TARGET_PROVINCE = "KwaZulu-Natal"

# Registration data period
START_YEAR = 2011
CURRENT_YEAR = 2026

print("Source:", IEC_VOTER_REGISTRATION_URL)
print("Province:", TARGET_PROVINCE)
print("Period:", START_YEAR, "to", CURRENT_YEAR)

Source: https://www.elections.org.za/pw/StatsData/Voter-Registration-Statistics
Province: KwaZulu-Natal
Period: 2011 to 2026


In [3]:
# Before extracting data, lets start by inspecting the IEC page
# to understand how the voter registration statistics
# are presented and where the actual data is located.

response = requests.get(
    IEC_VOTER_REGISTRATION_URL,
    timeout=30
)

print("HTTP status:", response.status_code)
print("Content type:", response.headers.get("Content-Type"))
print("Response size:", len(response.content), "bytes")

# Basic verification
assert response.status_code == 200, "IEC page could not be loaded."

print("IEC source loaded successfully.")

HTTP status: 200
Content type: text/html; charset=utf-8
Response size: 131362 bytes
IEC source loaded successfully.


In [4]:
# The page loaded successfully, so we now inspect
# its HTML structure to identify tables, links,
# downloads, or other elements containing the data.

soup = BeautifulSoup(response.text, "html.parser")

# Find tables on the page
tables = soup.find_all("table")

# Find links on the page
links = soup.find_all("a")

print("Tables found:", len(tables))
print("Links found:", len(links))

print("\nFirst 20 links:")
for i, link in enumerate(links[:20], start=1):
    text = link.get_text(" ", strip=True)
    href = link.get("href")

    print(f"{i}. {text} -> {href}")

Tables found: 1
Links found: 136

First 20 links:
1.  -> ../
2. I want to -> #
3. Register to vote now -> https://registertovote.elections.org.za
4. Apply for a special vote -> https://registertovote.elections.org.za/voter/specialvotes/application/submit
5. Check status of my special vote application -> https://registertovote.elections.org.za/voter/specialvotes/application/status
6. Check my voter registration status -> ../Voter/Voter-Information
7. See where my voting station is -> http://maps.elections.org.za/vsfinder/
8. Who is my ward councillor -> ../Voter/Who-Is-My-Ward-Councillor
9. Create my IEC voter portal profile -> https://registertovote.elections.org.za/account/create
10. Do business with the IEC -> ../About-Us/Auctions-And-Tenders
11. Voters -> #
12. Register to vote -> https://registertovote.elections.org.za
13. Check my voter registration status -> ../Voter/Voter-Information
14. Apply for a special vote -> https://registertovote.elections.org.za/voter/specialvotes/appli

In [5]:
# The page contains one HTML table. We inspect its
# headers and first few rows to determine whether
# it contains the voter registration data we need.

table = tables[0]

# Extract table headers
headers = [
    cell.get_text(" ", strip=True)
    for cell in table.find_all("th")
]

# Extract table rows
rows = table.find_all("tr")

print("Table headers:", headers)
print("Number of rows:", len(rows))

print("\nFirst 10 rows:")
for i, row in enumerate(rows[:10], start=1):
    values = [
        cell.get_text(" ", strip=True)
        for cell in row.find_all(["th", "td"])
    ]
    print(f"{i}: {values}")

Table headers: ['', '', '', '', '', '', '', '', '', '', '', '']
Number of rows: 9

First 10 rows:
1: ['Eastern Cape', '3,627,705', '12.41%']
2: ['Free State', '1,533,026', '5.25%']
3: ['Gauteng', '6,749,688', '23.1%']
4: ['Kwazulu-natal', '6,028,879', '20.63%']
5: ['Mpumalanga', '2,169,838', '7.43%']
6: ['Northern Cape', '713,727', '2.44%']
7: ['Limpopo', '3,036,959', '10.39%']
8: ['North West', '1,907,002', '6.53%']
9: ['Western Cape', '3,456,318', '11.83%']


In [6]:
# The visible table only provides provincial totals.
# We therefore inspect the page source for references
# to APIs, data files, scripts, or other endpoints that
# may contain the detailed registration statistics.

page_text = response.text

keywords = [
    "api",
    "voter",
    "registration",
    "statistics",
    "municipality",
    "ward",
    "json",
    "ajax"
]

print("Relevant references found in page source:\n")

for keyword in keywords:
    count = page_text.lower().count(keyword.lower())
    print(f"{keyword}: {count}")

Relevant references found in page source:

api: 3
voter: 101
registration: 12
statistics: 16
municipality: 2
ward: 8
json: 0
ajax: 5


In [7]:
# The page source contains API and AJAX references.
# We now print the surrounding source code so we can
# identify the endpoint used to obtain detailed data.

lines = page_text.splitlines()

keywords_to_find = [
    "api",
    "ajax",
    "municipality",
    "ward"
]

for line_number, line in enumerate(lines, start=1):
    line_lower = line.lower()

    if any(keyword in line_lower for keyword in keywords_to_find):
        print(f"\n--- Line {line_number} ---")
        print(line.strip())


--- Line 50 ---
|| /1207|6310|6590|3gso|4thp|50[1-6]i|770s|802s|a wa|abac|ac(er|oo|s\-)|ai(ko|rn)|al(av|ca|co)|amoi|an(ex|ny|yw)|aptu|ar(ch|go)|as(te|us)|attw|au(di|\-m|r |s )|avan|be(ck|ll|nq)|bi(lb|rd)|bl(ac|az)|br(e|v)w|bumb|bw\-(n|u)|c55\/|capi|ccwa|cdm\-|cell|chtm|cldc|cmd\-|co(mp|nd)|craw|da(it|ll|ng)|dbte|dc\-s|devi|dica|dmob|do(c|p)o|ds(12|\-d)|el(49|ai)|em(l2|ul)|er(ic|k0)|esl8|ez([4-7]0|os|wa|ze)|fetc|fly(\-|_)|g1 u|g560|gene|gf\-5|g\-mo|go(\.w|od)|gr(ad|un)|haie|hcit|hd\-(m|p|t)|hei\-|hi(pt|ta)|hp( i|ip)|hs\-c|ht(c(\-| |_|a|g|p|s|t)|tp)|hu(aw|tc)|i\-(20|go|ma)|i230|iac( |\-|\/)|ibro|idea|ig01|ikom|im1k|inno|ipaq|iris|ja(t|v)a|jbro|jemu|jigs|kddi|keji|kgt( |\/)|klon|kpt |kwc\-|kyo(c|k)|le(no|xi)|lg( g|\/(k|l|u)|50|54|\-[a-w])|libw|lynx|m1\-w|m3ga|m50\/|ma(te|ui|xo)|mc(01|21|ca)|m\-cr|me(rc|ri)|mi(o8|oa|ts)|mmef|mo(01|02|bi|de|do|t(\-| |o|v)|zz)|mt(50|p1|v )|mwbp|mywa|n10[0-2]|n20[2-3]|n30(0|2)|n50(0|2|5)|n7(0(0|1)|10)|ne((c|m)\-|on|tf|wf|wg|wt)|nok(6|i)|nzph|o2im|op(ti|wv)|o

In [8]:
# this output actually gives us the important clue Search and view voter registration statistics by selecting province and municipality below.
# The IEC page requires a province and municipality
# selection before showing detailed registration data.
# We inspect the form controls so we can reproduce
# that process programmatically.

selects = soup.find_all("select")

print("Dropdowns found:", len(selects))

for i, select in enumerate(selects, start=1):
    print(f"\n--- Dropdown {i} ---")
    print("Name:", select.get("name"))
    print("ID:", select.get("id"))

    options = select.find_all("option")

    print("Number of options:", len(options))

    for option in options[:15]:
        print(
            "Value:",
            option.get("value"),
            "| Text:",
            option.get_text(" ", strip=True)
        )

Dropdowns found: 2

--- Dropdown 1 ---
Name: ctl00$MainContent$ddlProvinces
ID: MainContent_ddlProvinces
Number of options: 10
Value: -1 | Text: All provinces
Value: 1 | Text: Eastern Cape
Value: 2 | Text: Free State
Value: 3 | Text: Gauteng
Value: 4 | Text: KwaZulu-Natal
Value: 7 | Text: Limpopo
Value: 5 | Text: Mpumalanga
Value: 8 | Text: North West
Value: 6 | Text: Northern Cape
Value: 9 | Text: Western Cape

--- Dropdown 2 ---
Name: ctl00$MainContent$ddlMunicipalities
ID: MainContent_ddlMunicipalities
Number of options: 1
Value: -1 | Text: All municipalities


In [9]:
# The municipality list currently contains only
# "All municipalities". We therefore inspect the
# province dropdown to determine what happens when
# a province is selected.
#
# This allows us to retrieve the municipality list
# directly from the IEC rather than hard-coding it.

province_select = soup.find(
    "select",
    {"name": "ctl00$MainContent$ddlProvinces"}
)

print("Province dropdown found:", province_select is not None)

print("\nProvince dropdown HTML:")
print(province_select.prettify())

Province dropdown found: True

Province dropdown HTML:
<select class="form-control col" id="MainContent_ddlProvinces" name="ctl00$MainContent$ddlProvinces">
 <option selected="selected" value="-1">
  All provinces
 </option>
 <option value="1">
  Eastern Cape
 </option>
 <option value="2">
  Free State
 </option>
 <option value="3">
  Gauteng
 </option>
 <option value="4">
  KwaZulu-Natal
 </option>
 <option value="7">
  Limpopo
 </option>
 <option value="5">
  Mpumalanga
 </option>
 <option value="8">
  North West
 </option>
 <option value="6">
  Northern Cape
 </option>
 <option value="9">
  Western Cape
 </option>
</select>



In [10]:
# The province dropdown has no inline onchange event.
# We therefore search the page source for references
# to the province and municipality controls to identify
# how the IEC populates municipalities.

control_names = [
    "ddlProvinces",
    "ddlMunicipalities"
]

for control in control_names:
    print(f"\n{'=' * 60}")
    print(f"References to: {control}")
    print(f"{'=' * 60}")

    for line_number, line in enumerate(lines, start=1):
        if control.lower() in line.lower():
            print(f"--- Line {line_number} ---")
            print(line.strip())


References to: ddlProvinces
--- Line 885 ---
<select name="ctl00$MainContent$ddlProvinces" id="MainContent_ddlProvinces" class="form-control col">

References to: ddlMunicipalities
--- Line 912 ---
<select name="ctl00$MainContent$ddlMunicipalities" id="MainContent_ddlMunicipalities" class="form-control col">


In [11]:
# The province and municipality controls do not expose
# their interaction directly in the HTML. Since the IEC
# page uses ASP.NET Web Forms, we inspect the form action
# and method before reproducing the selection process.

form = soup.find("form")

print("Form found:", form is not None)

if form is not None:
    print("Form action:", form.get("action"))
    print("Form method:", form.get("method"))

    print("\nForm controls:")
    
    controls = form.find_all(["input", "select", "button"])

    for control in controls:
        print(
            control.name,
            "| name:", control.get("name"),
            "| id:", control.get("id"),
            "| type:", control.get("type"),
            "| value:", control.get("value")
        )

Form found: True
Form action: ./Voter-Registration-Statistics
Form method: post

Form controls:
input | name: __VIEWSTATE | id: __VIEWSTATE | type: hidden | value: H64l6VEOUGmjHsRtr2cLu8/H+6vuOO/aN5PDqqesoIsXx/c57DZK1d0uTKzxXwhCKADDylct3vT5mNsKOFSMMeUdTs6t6MvNZwjrHeCP+Eav/CxnNNcjr62x4WE3CjT4XaTpfGBXTKUARuV28JvJw42YmWR2Tl3pw2sxJUgz4kpgwv/580l/TOcpjg6cfmUBQbTv+wVFPkhTrinTiMMehsaQquBT8eI1+MqoJkEfYZ1Yr/Bz7WgIjM7S8TvvEI6n1M+CvFD+1zekPxlQksEWtCieJH20JRoRKaxn88kohmR8t4PP7YL6PxF0XPXpcU6mbvAePwL7/uXhCmOG3HwLO16g8pEI9i7OLcUKda7YcZnBx3Tde/8u6Y/ml1J96TpcKHj44QWf8JCSMXdveI4yINd31lfL7o2GPPUUrzf379dEVcR/GJp1igXZKS3aRt7qudJV3gAQskaswScfwhszjw9a95IukznAzjzmq48Eqoxu7oiUkI6oSoFoLUAdeOzSyzBJbMYjVbAo7KPbpspcBJ6v65bIhy8sot7X09UQITDqbkC9v+cA8Pec83VLUlMRZsonTj8J3nVIOpEIEUTvwNxoo7TQwKwmLcwLc3z/YveTU85pQC0eeMXpoDTiqk7jSEJniNc5X+QdWmMsFehPk8CrwtbO12hitXz5TxI5rqToWN1cwbWRsd47c5Q6TZwVezxWuAW/2iKlavaJXC5DSbtasnk546AHyzqH6oIi7iiPcrgs0Pa4OwRpQ2cxF8+PJL21XFdio6hxokqIAQ3iCxE2zwWJw2FpLDhnmF+c/2jbnWzADtz1Fa5VSp9nxY2Z2kXv

In [12]:
# lets look into the search logic now since municipality drobdowns have no submit they just update upone selection and show whats selected, The form contains a search button and province-specific
# voter-count buttons. We inspect the JavaScript references
# associated with these controls to determine how the IEC
# retrieves the detailed registration statistics.

search_ids = [
    "uxSearchButton",
    "btnVoterCountKZN",
    "ddlProvinces",
    "ddlMunicipalities"
]

for search_id in search_ids:
    print(f"\n{'=' * 60}")
    print(f"Searching JavaScript references for: {search_id}")
    print(f"{'=' * 60}")

    for line_number, line in enumerate(lines, start=1):
        if search_id.lower() in line.lower():
            print(f"--- Line {line_number} ---")
            print(line.strip())


Searching JavaScript references for: uxSearchButton
--- Line 368 ---
<button onclick="__doPostBack('ctl00$uxSearchButton','')" id="uxSearchButton" class="btn btn-outline-success my-2 my-sm-0" type="button" style="margin-right: 5px; margin-left: 10px;"></button>

Searching JavaScript references for: btnVoterCountKZN
--- Line 1250 ---
<button onclick="__doPostBack('ctl00$MainContent$btnVoterCountKZN','')" id="MainContent_btnVoterCountKZN" type="button" class="btn btn-md btn-link font-weight-bold" ProvID="4">

Searching JavaScript references for: ddlProvinces
--- Line 885 ---
<select name="ctl00$MainContent$ddlProvinces" id="MainContent_ddlProvinces" class="form-control col">

Searching JavaScript references for: ddlMunicipalities
--- Line 912 ---
<select name="ctl00$MainContent$ddlMunicipalities" id="MainContent_ddlMunicipalities" class="form-control col">


In [13]:
# The IEC uses ASP.NET __doPostBack() for its buttons.
# The municipality dropdown may be populated by JavaScript,
# so we inspect the JavaScript files loaded by the page
# before attempting to reproduce the requests ourselves.

scripts = soup.find_all("script")

script_urls = []

for script in scripts:
    src = script.get("src")

    if src:
        script_urls.append(src)

print("JavaScript files found:", len(script_urls))

print("\nJavaScript files:")
for i, src in enumerate(script_urls, start=1):
    print(f"{i}. {src}")

JavaScript files found: 12

JavaScript files:
1. https://cdn.jsdelivr.net/sharer.js/latest/sharer.min.js
2. https://www.googletagmanager.com/gtag/js?id=G-J6PHHGXBL0
3. /pw/bundles/modernizr?v=inCVuEFe6J4Q07A0AcRsbJic_UE5MwpRMNGcOtk94TE1
4. https://code.jquery.com/jquery-1.12.4.js
5. https://code.jquery.com/ui/1.12.1/jquery-ui.js
6. https://ajax.googleapis.com/ajax/libs/jquery/3.5.1/jquery.min.js
7. https://kit.fontawesome.com/a076d05399.js
8. https://use.fontawesome.com/2afe1f08ec.js
9. /pw/bundles/MsAjaxJs?v=D6VN0fHlwFSIWjbVzi6mZyE9Ls-4LNrSSYVGRU46XF81
10. ../Scripts/jquery-3.5.0.min.js
11. ../Scripts/bootstrap.min.js
12. /pw/bundles/WebFormsJs?v=N8tymL9KraMLGAMFuPycfH3pXe6uUlRXdhtYv8A_jUU1


In [14]:
# The IEC page uses ASP.NET Web Forms.
# We inspect the WebForms JavaScript bundles to determine
# whether the province and municipality dropdowns use
# AJAX or another client-side mechanism.

base_url = "https://www.elections.org.za"

webforms_scripts = [
    src for src in script_urls
    if "MsAjaxJs" in src or "WebFormsJs" in src
]

print("WebForms scripts found:", len(webforms_scripts))

for i, script_url in enumerate(webforms_scripts, start=1):
    if script_url.startswith("/"):
        full_url = base_url + script_url
    elif script_url.startswith("../"):
        full_url = base_url + "/pw/" + script_url.replace("../", "")
    else:
        full_url = script_url

    print(f"\n{'=' * 60}")
    print(f"Script {i}")
    print(full_url)
    print(f"{'=' * 60}")

    script_response = requests.get(
        full_url,
        timeout=30
    )

    print("HTTP status:", script_response.status_code)
    print("Content type:", script_response.headers.get("Content-Type"))
    print("Size:", len(script_response.content), "bytes")

    script_text = script_response.text

    # Look for the controls we are interested in
    search_terms = [
        "ddlProvinces",
        "ddlMunicipalities",
        "btnVoterCountKZN",
        "__doPostBack"
    ]

    for term in search_terms:
        print(
            f"{term}:",
            script_text.lower().count(term.lower())
        )

WebForms scripts found: 2

Script 1
https://www.elections.org.za/pw/bundles/MsAjaxJs?v=D6VN0fHlwFSIWjbVzi6mZyE9Ls-4LNrSSYVGRU46XF81
HTTP status: 200
Content type: text/javascript; charset=utf-8
Size: 145442 bytes
ddlProvinces: 0
ddlMunicipalities: 0
btnVoterCountKZN: 0
__doPostBack: 6

Script 2
https://www.elections.org.za/pw/bundles/WebFormsJs?v=N8tymL9KraMLGAMFuPycfH3pXe6uUlRXdhtYv8A_jUU1
HTTP status: 200
Content type: text/javascript; charset=utf-8
Size: 61394 bytes
ddlProvinces: 0
ddlMunicipalities: 0
btnVoterCountKZN: 0
__doPostBack: 2


In [15]:
# The standard WebForms bundles do not reference the
# IEC-specific dropdown controls. We therefore inspect
# inline JavaScript embedded directly in this page.
#
# We are looking specifically for logic that handles:
# - province selection
# - municipality loading
# - the search button
# - AJAX requests

inline_scripts = soup.find_all("script", src=False)

print("Inline script blocks found:", len(inline_scripts))

search_terms = [
    "ddlProvinces",
    "ddlMunicipalities",
    "uxSearchButton",
    "btnVoterCountKZN",
    "ajax",
    "$.post",
    "$.ajax",
    "getJSON",
    "__doPostBack"
]

for i, script in enumerate(inline_scripts, start=1):
    script_text = script.get_text()

    if not script_text.strip():
        continue

    matched_terms = [
        term for term in search_terms
        if term.lower() in script_text.lower()
    ]

    if matched_terms:
        print(f"\n{'=' * 70}")
        print(f"INLINE SCRIPT {i}")
        print("Matched:", matched_terms)
        print(f"{'=' * 70}")

        print(script_text[:10000])

Inline script blocks found: 8


In [16]:
# None of the inline scripts directly referenced the
# dropdown controls by name. We now inspect their sizes
# so we can identify which scripts contain page-specific
# logic before examining their contents.

print("Inline script blocks:")

for i, script in enumerate(inline_scripts, start=1):
    script_text = script.get_text()

    print(
        f"{i}.",
        "Characters:", len(script_text),
        "|",
        "Non-empty:", bool(script_text.strip())
    )

Inline script blocks:
1. Characters: 3815 | Non-empty: True
2. Characters: 346 | Non-empty: True
3. Characters: 190 | Non-empty: True
4. Characters: 372 | Non-empty: True
5. Characters: 634 | Non-empty: True
6. Characters: 1399 | Non-empty: True
7. Characters: 58 | Non-empty: True
8. Characters: 865 | Non-empty: True


In [17]:
# The inline scripts are small, so we inspect their
# contents directly. This should reveal whether the IEC
# page contains any page-specific AJAX or selection logic
# that was not visible from the HTML controls.

for i, script in enumerate(inline_scripts, start=1):
    script_text = script.get_text().strip()

    print(f"\n{'=' * 70}")
    print(f"INLINE SCRIPT {i} — {len(script_text)} characters")
    print(f"{'=' * 70}")

    print(script_text)


INLINE SCRIPT 1 — 3793 characters
//var _gaq = _gaq || [];
        //_gaq.push(['_setAccount', 'UA-6489677-1']);
        //_gaq.push(['_trackPageview']);

        //(function () {
        //    var ga = document.createElement('script'); ga.type = 'text/javascript'; ga.async = true;
        //    ga.src = ('https:' == document.location.protocol ? 'https://ssl' : 'http://www') + '.google-analytics.com/ga.js';
        //    var s = document.getElementsByTagName('script')[0]; s.parentNode.insertBefore(ga, s);
        //})();

        document.addEventListener("DOMContentLoaded", function (event) {

            // Uses sharer.js 
            //  https://ellisonleao.github.io/sharer.js/#twitter    
            var url = window.location.href;
            var title = document.title;
            var subject = "Read this article from IEC website ";
            var via = "IECSouthAfrica";

            //facebook
            $('#share-fb').attr('data-url', url).attr('data-sharer', 'facebook');
  